# Rescue Drone: causal environment draft

This notebook mirrors the presentation flow of the Lunar Lander example while making the causal semantics explicit. It demonstrates three distinct interactions with the same environment: **see** (natural pilot), **do** (replace the action mechanism), and **ctf_do** (intercept the intended action and replace it while preserving the same stage-level exogenous realization).

Textbook anchors: SCM Definition 2.1.1 (p. 44), intervention/submodel and effectiveness Definitions 2.2.2-2.2.5 (pp. 48-49), causal diagram Definition 2.4.1 (pp. 61-63), causal decision model Definition 8.1.3 (p. 532), CRL task Definition 8.1.4 (p. 539), and confounded MDP environment Definition 9.4.4 (p. 623).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if repo_root.name == 'examples':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from causal_gym.core import Task
from causal_gym.envs.rescue_drone import RescueDronePCH

ACTION_NAMES = {0: 'hover', 1: 'up', 2: 'right', 3: 'down', 4: 'left'}

def make_env(regime='see_do', *, debug=False, render_mode='rgb_array'):
    return RescueDronePCH(
        task=Task(learning_regime=regime, assumptions='dag'),
        render_mode=render_mode,
        include_latent_in_info=debug,
    )

def observation_only_greedy_policy(obs):
    dx = float(obs[3] - obs[0])
    dy = float(obs[4] - obs[1])
    if dx == 0 and dy == 0:
        return 0
    if abs(dx) > abs(dy):
        return 2 if dx > 0 else 4
    return 3 if dy > 0 else 1

print('Imports and policies ready.')

## 1. The executable SCM

For each stage, the draft implements $U_i=(G_i,H_i)$, $O_i\leftarrow f_O(S_i)$, $X_i\leftarrow f_X(O_i,U_i)$, $S_{i+1}\leftarrow f_S(S_i,X_i,U_i)$, and $Y_i\leftarrow f_Y(S_i,X_i,U_i)$. The learner sees the six-dimensional observation but not gust/hazard. The natural pilot can sense the latent stage condition, so $U_i$ is a genuine common cause of its action and mission outcome.

In [ ]:
env = make_env('see_do', debug=False)
obs, reset_info = env.reset(seed=7)
graph = env.get_graph

print('Observation:', np.round(obs, 3))
print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
print('Graph nodes:', [node['name'] for node in graph.nodes])
print('Graph edges:')
for edge in graph.edges:
    symbol = '<->' if edge['type_'] == 'bidirected' else '->'
    print(f"  {edge['from_']} {symbol} {edge['to_']}")
assert 'exogenous' not in reset_info

## 2. L1 versus L2: seeing is not doing

`see()` lets the natural action mechanism choose the action. `do(policy)` replaces that mechanism with a learner policy. The comparison below uses matched seeds; it is a smoke demonstration of different data-generating regimes, not a policy-quality benchmark.

In [ ]:
def run_episode(seed, regime):
    local_env = make_env('see_do', render_mode=None)
    obs, _ = local_env.reset(seed=seed)
    total_reward = 0.0
    steps = 0
    last_info = {}
    while True:
        if regime == 'see':
            obs, reward, terminated, truncated, last_info = local_env.see()
        elif regime == 'do':
            obs, reward, terminated, truncated, last_info = local_env.do(observation_only_greedy_policy)
        else:
            raise ValueError(regime)
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    local_env.close()
    return {
        'seed': seed,
        'regime': regime,
        'reward': total_reward,
        'steps': steps,
        'rescued': bool(last_info.get('rescued', False)),
        'end_reason': last_info.get('end_reason'),
    }

results = [run_episode(seed, regime) for seed in range(30) for regime in ('see', 'do')]
for regime in ('see', 'do'):
    subset = [row for row in results if row['regime'] == regime]
    success = np.mean([row['rescued'] for row in subset])
    reward = np.mean([row['reward'] for row in subset])
    print(f"{regime:>3}: success={success:.2f}, mean return={reward:.3f}")

## 3. L3 diagnostic: the same unit, multiple actions

A counterfactual comparison must hold the unit-level exogenous realization fixed. `unit_counterfactuals()` evaluates every candidate action from the same state with the same $U_i$. Then `ctf_do()` intercepts the natural intention and executes a selected alternative without resampling $U_i$. This supports a one-step, intention-conditioned simulator demonstration; it is not a claim that arbitrary retrospective counterfactuals are identifiable from logged data.

In [ ]:
ctf_env = make_env('ctf_do', debug=True)
ctf_env.reset(
    seed=11,
    options={'drone_position': (3, 5), 'victim_position': (3, 1)},
)
table = ctf_env.unit_counterfactuals()
print('Fixed exogenous unit:', table['exogenous'])
print('Natural intention:', table['intended_action_name'])
for action, outcome in table['alternatives'].items():
    next_state = outcome['next_state']
    print(
        f"{action} {outcome['action_name']:<5} -> "
        f"position=({next_state['drone_x']}, {next_state['drone_y']}), "
        f"reward={outcome['reward']:.3f}"
    )

best_action = max(table['alternatives'], key=lambda a: table['alternatives'][a]['reward'])
_, reward, terminated, truncated, info = ctf_env.ctf_do(lambda obs, intended: best_action)
assert info['exogenous'] == table['exogenous']
print(
    'Executed:', ACTION_NAMES[info['action']],
    '| intended:', ACTION_NAMES[info['natural_action']],
    '| reward:', round(reward, 3),
)

In [ ]:
frame = ctf_env.render(show_wind=True, show_natural_action=True)
plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.title('Rescue Drone after one intention-conditioned intervention')
plt.axis('off')
plt.show()
ctf_env.close()

## 4. What this draft establishes - and what it does not

**Established by code:** the environment has explicit state/action/reward mechanisms; stage noise is seeded and sampled once; the natural action and transition/reward share that same noise; `do()` replaces only the action mechanism; the causal graph reflects the implemented arguments; and a paired one-step counterfactual can reuse the same unit.

**Still open for research review:** whether gust/hazard are the scientifically right confounders; whether the observation is sufficient for a Markov policy; whether reward shaping matches the intended rescue objective; how persistent spatial weather should be modeled without silently turning the task into a POMDP; and which estimand/algorithm the environment is meant to benchmark.